# Аналіз сигналів ЕЕГ: Виділення Альфа-ритму та очищення від артефактів

## 1. Детальний опис умови

Цей проект присвячений обробці та аналізу реальних біомедичних сигналів — електроенцефалограми (ЕЕГ). Основним завданням є виявлення **Альфа-ритму** (коливання в діапазоні 8–13 Гц), який найчіткіше проявляється у потиличній зоні кори головного мозку під час стану спокою із закритими очима.

**Вхідні дані:**
- Набір даних: PhysioNet EEGBCI (через бібліотеку MNE).
- Суб'єкт: Subject 1.
- Записи (Runs): 
    - Run 1: Спокій з відкритими очима.
    - Run 2: Спокій із закритими очима.
- Кількість каналів: 64 електроди.
- Частота дискретизації: 160 Гц.

## 2. Опис змінних та математичних формул

Для аналізу ми використовуємо метод головних компонент (PCA) для просторової фільтрації та швидке перетворення Фур'є (FFT) для частотного аналізу.

**Основні змінні:**
- $X$: Матриця центрованих даних розмірністю $(T \times C)$, де $T$ — кількість часових відліків, $C$ — кількість каналів.
- $C_{ov}$: Матриця просторової коваріації $(64 \times 64)$.
- $\lambda, V$: Власні числа та відповідні власні вектори.

**Математичні етапи:**
1. **Центрування:** $X_{centered} = X - \mu_X$, де $\mu_X$ — вектор середніх значень каналів.
2. **Коваріаційна матриця:** $C_{ov} = \frac{1}{N-1} X^T X$.
3. **Власна декомпозиція:** Розв'язання рівняння $C_{ov} V = V \Lambda$.
4. **PCA Проекція:** Перехід у простір компонент $Z = X V$.
5. **Фільтрація:** Видалення компонент, чиї власні числа $\lambda > 10 \cdot median(\lambda)$ (пошук артефактів кліпання).
6. **Зворотна проекція:** Повернення очищених даних у фізичний простір каналів: $\hat{X} = Z_{filtered} V^T$.
7. **FFT:** Обчислення спектральної потужності: $P(f) = |FFT(\hat{X}_{O1})|^2$.

In [ ]:
# Step 0: Setup environment in SageMath
import sys
import numpy as np
try:
    import mne
except ImportError:
    !{sys.executable} -m pip install mne
    import mne

from mne.datasets import eegbci

In [ ]:
# Step 1: Data Loading & Preprocessing
subjects = [1]
runs = [2] # Run 2 = Eyes Closed (Alpha wave search)

edf_files = eegbci.load_data(subjects, runs)
raw = mne.io.read_raw_edf(edf_files[0], preload=True)
mne.datasets.eegbci.standardize(raw)

# Apply digital filter to remove DC drift and high-freq noise
raw.filter(1., 40., fir_design='firwin')
raw.pick_types(eeg=True)

# Convert to SageMath matrix
raw_data = raw.get_data().T # Shape: (Time, Channels)
X_raw = matrix(RDF, raw_data)

In [ ]:
# Step 2: PCA Artifact Removal (Spatial Filtering)

# Centering
col_means = [col.mean() for col in X_raw.columns()]
mean_matrix = matrix(RDF, X_raw.nrows(), X_raw.ncols(), lambda i, j: col_means[j])
X_centered = X_raw - mean_matrix

# Covariance matrix (64x64)
cov_matrix = (X_centered.transpose() * X_centered) / (X_raw.nrows() - 1)

# Eigen-decomposition
D, P = cov_matrix.eigenmatrix_right()
eig_vals = D.diagonal()
eig_pairs = sorted(zip(eig_vals, P.columns()), key=lambda x: x[0])

eigenvalues = [p[0] for p in eig_pairs]
eigenvectors = column_matrix(RDF, [p[1] for p in eig_pairs])

# Thresholding
threshold = np.median(eigenvalues) * 10
good_components_count = len([v for v in eigenvalues if v <= threshold])

# Project to PCA base and clean
data_pca = X_centered * eigenvectors

# We only zero out if artifacts are actually present
for i in range(good_components_count, len(eigenvalues)):
    data_pca[:, i] = 0

# Project back to channel space
X_cleaned = data_pca * eigenvectors.transpose()

In [ ]:
# Step 3: Frequency Analysis (FFT) & Visualization

ch_names = raw.info['ch_names']
target_id = ch_names.index('O1') # Occipital 1 channel
fs = raw.info['sfreq']

target_signal = list(X_cleaned.column(target_id))
N = len(target_signal)

# FFT calculation
fft_vals = np.fft.rfft(target_signal)
fft_freqs = np.fft.rfftfreq(N, 1/fs)
fft_power = np.abs(fft_vals)**2

# SageMath Plotting
plot_points = [(f, p) for f, p in zip(fft_freqs, fft_power) if f <= 40]
spectrum_plot = line(plot_points, color='darkred', legend_label="O1 Spectrum")

# Expected Alpha line
max_p = max([p for f, p in plot_points])
alpha_line = line([(10, 0), (10, max_p)], color='gray', linestyle='--', legend_label="10Hz Alpha")

(spectrum_plot + alpha_line).show(axes_labels=['Freq (Hz)', 'Power'], title="Результат FFT (Канал O1)", gridlines=True)